# 04_feature_selection

## 1.构建融合特征

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

In [28]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FUSION_DIR = DATA_DIR /"fusion_selected"
OUTPUT_DIR = DATA_DIR /"fusion_selected" /"fusion_selected"

FUSION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(DATA_DIR)
print(FUSION_DIR)
print(OUTPUT_DIR)

d:\360Downloads\bioinformatics\Task\AIP
d:\360Downloads\bioinformatics\Task\AIP\data\processed
d:\360Downloads\bioinformatics\Task\AIP\data\processed\fusion_selected
d:\360Downloads\bioinformatics\Task\AIP\data\processed\fusion_selected\fusion_selected


In [29]:
# 读取基础特征

# handcrafted selected
X_train_hand = np.load(DATA_DIR / "handcrafted" / "X_train_handcrafted.npy")
X_test_hand = np.load(DATA_DIR /  "handcrafted" / "X_test_handcrafted.npy")
y_train = np.load(DATA_DIR / "handcrafted" / "y_train.npy")
y_test = np.load(DATA_DIR / "handcrafted" / "y_test.npy")

# ProtT5
X_train_prott5 = np.load(DATA_DIR / "prott5" / "X_train_prott5.npy")
X_test_prott5 = np.load(DATA_DIR / "prott5" / "X_test_prott5.npy")

# ESM2
X_train_esm2 = np.load(DATA_DIR / "esm2" / "X_train_esm2.npy")
X_test_esm2 = np.load(DATA_DIR / "esm2" / "X_test_esm2.npy")

In [31]:
# 构建融合特征
def concatenate_features(*arrays):
    return np.concatenate(arrays, axis=1)

feature_sets = {}

feature_sets["handcrafted_prott5"] = (
    concatenate_features(X_train_hand, X_train_prott5),
    concatenate_features(X_test_hand, X_test_prott5)
)

feature_sets["handcrafted_esm2"] = (
    concatenate_features(X_train_hand, X_train_esm2),
    concatenate_features(X_test_hand, X_test_esm2)
)

feature_sets["prott5_esm2"] = (
    concatenate_features(X_train_prott5, X_train_esm2),
    concatenate_features(X_test_prott5, X_test_esm2)
)

feature_sets["all_fusion"] = (
    concatenate_features(X_train_hand, X_train_prott5, X_train_esm2),
    concatenate_features(X_test_hand, X_test_prott5, X_test_esm2)
)

for name, (Xtr, Xte) in feature_sets.items():
    print(name, Xtr.shape, Xte.shape)

handcrafted_prott5 (3583, 1209) (897, 1209)
handcrafted_esm2 (3583, 921) (897, 921)
prott5_esm2 (3583, 1248) (897, 1248)
all_fusion (3583, 1689) (897, 1689)


In [32]:
# 保存 raw fusion 版本
def save_feature_version(save_dir, X_train, X_test, y_train, y_test):
    save_dir.mkdir(parents=True, exist_ok=True)
    np.save(save_dir / "X_train.npy", X_train)
    np.save(save_dir / "X_test.npy", X_test)
    np.save(save_dir / "y_train.npy", y_train)
    np.save(save_dir / "y_test.npy", y_test)

for name, (Xtr, Xte) in feature_sets.items():
    save_feature_version(FUSION_DIR / name, Xtr, Xte, y_train, y_test)

## 2.特征筛选方法

### （1）PCA

In [33]:
def apply_pca(X_train, X_test, n_components=0.95):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    return X_train_pca, X_test_pca, pca, scaler

### （2）LASSO

In [34]:
def apply_lasso_selection(X_train, X_test, y_train, C=0.1, threshold="mean"):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    selector = SelectFromModel(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            random_state=42,
            max_iter=2000
        ),
        threshold=threshold
    )

    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)

    return X_train_sel, X_test_sel, selector, scaler

### （3）LightGBM

In [35]:
def apply_lgbm_selection(X_train, X_test, y_train, threshold="mean"):
    selector = SelectFromModel(
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1
        ),
        threshold=threshold
    )

    X_train_sel = selector.fit_transform(X_train, y_train)
    X_test_sel = selector.transform(X_test)

    return X_train_sel, X_test_sel, selector

In [36]:
# 批量处理所有融合特征
# 对每个feature set 都采用上述三种方法
summary_rows = []

for feature_name, (Xtr, Xte) in feature_sets.items():
    # 原始版本
    summary_rows.append([feature_name, "raw", Xtr.shape[1], Xte.shape[1]])

    # PCA
    Xtr_pca, Xte_pca, pca_model, pca_scaler = apply_pca(Xtr, Xte, n_components=0.95)
    save_feature_version(OUTPUT_DIR / f"{feature_name}_pca", Xtr_pca, Xte_pca, y_train, y_test)
    summary_rows.append([feature_name, "pca", Xtr_pca.shape[1], Xte_pca.shape[1]])

    # LASSO
    Xtr_lasso, Xte_lasso, lasso_selector, lasso_scaler = apply_lasso_selection(
        Xtr, Xte, y_train, C=0.1, threshold="median"
    )
    save_feature_version(OUTPUT_DIR / f"{feature_name}_lasso", Xtr_lasso, Xte_lasso, y_train, y_test)
    summary_rows.append([feature_name, "lasso", Xtr_lasso.shape[1], Xte_lasso.shape[1]])

    # LGBM
    Xtr_lgbm, Xte_lgbm, lgbm_selector = apply_lgbm_selection(
        Xtr, Xte, y_train, threshold="median"
    )
    save_feature_version(OUTPUT_DIR / f"{feature_name}_lgbm", Xtr_lgbm, Xte_lgbm, y_train, y_test)
    summary_rows.append([feature_name, "lgbm", Xtr_lgbm.shape[1], Xte_lgbm.shape[1]])

[LightGBM] [Info] Number of positive: 1365, number of negative: 2218
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031363 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21184
[LightGBM] [Info] Number of data points in the train set: 3583, number of used features: 1191
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.380966 -> initscore=-0.485451
[LightGBM] [Info] Start training from score -0.485451


c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(
c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1365, number of negative: 2218
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032993 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 129002
[LightGBM] [Info] Number of data points in the train set: 3583, number of used features: 903
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.380966 -> initscore=-0.485451
[LightGBM] [Info] Start training from score -0.485451


c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(
c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1365, number of negative: 2218
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038183 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136982
[LightGBM] [Info] Number of data points in the train set: 3583, number of used features: 1248
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.380966 -> initscore=-0.485451
[LightGBM] [Info] Start training from score -0.485451


c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(
c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1365, number of negative: 2218
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 143584
[LightGBM] [Info] Number of data points in the train set: 3583, number of used features: 1671
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.380966 -> initscore=-0.485451
[LightGBM] [Info] Start training from score -0.485451


c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(
c:\Users\User\.conda\envs\ysy_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SelectFromModel was fitted with feature names
  warnings.warn(


In [26]:
# 输出summary表
summary_df = pd.DataFrame(
    summary_rows,
    columns=["feature_set", "selection_method", "train_dim", "test_dim"]
)

summary_df



,feature_set,selection_method,train_dim,test_dim
0,handcrafted_prott5,raw,1209,1209
1,handcrafted_prott5,pca,283,283
2,handcrafted_prott5,lasso,1209,1209
3,handcrafted_prott5,lgbm,1209,1209
4,handcrafted_esm2,raw,921,921
5,handcrafted_esm2,pca,378,378
6,handcrafted_esm2,lasso,921,921
7,handcrafted_esm2,lgbm,479,479
8,prott5_esm2,raw,1248,1248
9,prott5_esm2,pca,39,39


In [27]:

summary_df.to_csv(OUTPUT_DIR / "fusion_feature_selection_summary.csv", index=False)